# 09 — Model Artifact

## Objetivo
Guardar el modelo final entrenado y los metadatos necesarios para reutilizarlo en futuras predicciones, sin volver a entrenarlo desde cero.

Este notebook asume que el modelo seleccionado es HistGradientBoosting. Incluye dos alternativas:
- **Modelo completo:** usa todas las columnas predictoras.
- **Modelo compacto:** incorpora la selección de variables dentro de un `Pipeline`.

Elegí una alternativa después de revisar las métricas del notebook 07. Por defecto, se usa el modelo compacto, ya que reduce considerablemente la cantidad de variables con una pérdida pequeña de rendimiento.

## 1. Cargar los datos y preparar el conjunto de entrenamiento

Verificá que la ruta al dataset y el nombre del objetivo coincidan con los notebooks anteriores. El ejemplo usa `price` como variable objetivo y la misma división reproducible.

In [ ]:
import json
import joblib
import sklearn
import pandas as pd
import numpy as np

from datetime import datetime, timezone
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

DATA_PATH = Path("../data/processed/airbnb_dp.csv")
TARGET = "price"

df = pd.read_csv(DATA_PATH)

if TARGET not in df.columns:
    raise ValueError(
        f"No se encontró la columna objetivo {TARGET!r}. "
        f"Columnas disponibles: {df.columns.tolist()}"
    )

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Target:", TARGET)


## 2. Configurar el modelo elegido

En los experimentos previos, el modelo completo tuvo un rendimiento de test apenas superior. El compacto usa 43 variables aproximadamente, pero la selección exacta depende de los datos con los que se ajuste el selector.

La variable `MODEL_CHOICE` permite elegir cuál guardar. Si querés guardar el modelo compacto, dejá `"compact"`; para guardar el modelo completo, usá `"full"`.

In [ ]:
MODEL_CHOICE = "compact"  # opciones: "compact" o "full"

hgb_params = {
    "l2_regularization": 7.87,
    "learning_rate": 0.068,
    "max_iter": 140,
    "max_leaf_nodes": 38,
    "min_samples_leaf": 24,
    "random_state": 42
}

if MODEL_CHOICE == "compact":
    final_model = Pipeline([
        ("selector", SelectFromModel(
            estimator=ExtraTreesRegressor(
                n_estimators=100,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ),
            threshold="0.5*mean"
        )),
        ("model", HistGradientBoostingRegressor(**hgb_params))
    ])
elif MODEL_CHOICE == "full":
    final_model = HistGradientBoostingRegressor(**hgb_params)
else:
    raise ValueError("MODEL_CHOICE debe ser 'compact' o 'full'")


## 3. Entrenar el modelo final

Una vez decidida la configuración, entrenamos con el conjunto de entrenamiento. No usamos el test para ajustar el modelo.

In [ ]:
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

final_metrics = {
    "model_choice": MODEL_CHOICE,
    "n_input_features": int(X_train.shape[1]),
    "n_selected_features": (
        int(final_model.named_steps["selector"].get_support().sum())
        if MODEL_CHOICE == "compact"
        else int(X_train.shape[1])
    ),
    "test_r2": float(r2_score(y_test, y_pred)),
    "test_rmse": float(root_mean_squared_error(y_test, y_pred)),
    "test_mae": float(mean_absolute_error(y_test, y_pred))
}

pd.DataFrame([final_metrics]).round(4)


## 4. Guardar el artifact y sus metadatos

Guardamos el pipeline completo para que el selector y el modelo se apliquen en el orden correcto. También guardamos los nombres y el orden de las columnas de entrada, los parámetros, las métricas y la versión de scikit-learn.

**Seguridad:** los archivos `joblib`/pickle deben cargarse únicamente si provienen de una fuente confiable.

In [ ]:
ARTIFACT_DIR = Path("../models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

artifact_path = ARTIFACT_DIR / "airbnb_price_model.joblib"
metadata_path = ARTIFACT_DIR / "airbnb_price_model_metadata.json"

joblib.dump(final_model, artifact_path)

if MODEL_CHOICE == "compact":
    selected_columns = X_train.columns[
        final_model.named_steps["selector"].get_support()
    ].tolist()
else:
    selected_columns = X_train.columns.tolist()

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target": TARGET,
    "model_type": "HistGradientBoostingRegressor",
    "model_choice": MODEL_CHOICE,
    "input_columns": X_train.columns.tolist(),
    "selected_columns": selected_columns,
    "n_input_features": int(X_train.shape[1]),
    "n_selected_features": int(len(selected_columns)),
    "hyperparameters": hgb_params,
    "metrics": final_metrics,
    "sklearn_version": sklearn.__version__,
    "data_path": str(DATA_PATH)
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Modelo guardado en: {artifact_path.resolve()}")
print(f"Metadatos guardados en: {metadata_path.resolve()}")


## 5. Recargar el artifact y verificar que funciona

Esta comprobación carga el archivo guardado y verifica que las predicciones coincidan con las del modelo que está en memoria.

In [ ]:
loaded_model = joblib.load(artifact_path)
loaded_predictions = loaded_model.predict(X_test)

same_predictions = np.allclose(y_pred, loaded_predictions)

print("Predicciones reproducibles:", same_predictions)
print("Primeras 5 predicciones:", loaded_predictions[:5])

assert same_predictions, "Las predicciones cargadas no coinciden con el modelo original."


## 6. Cómo usar el modelo en otro script

El ejemplo siguiente muestra cómo cargar el modelo y predecir usando un DataFrame que contenga las mismas columnas de entrada que se utilizaron al entrenar. Los datos nuevos deben pasar por las mismas transformaciones previas al entrenamiento; si el dataset requiere limpieza o codificación adicional, esas operaciones también deben incluirse en un pipeline de preprocesamiento reutilizable.

In [ ]:
# Ejemplo de uso en un script independiente:
#
# import joblib
# import pandas as pd
#
# model = joblib.load("../models/airbnb_price_model.joblib")
# new_data = pd.DataFrame([{
#     # Completar con TODAS las columnas esperadas por el modelo.
# }])
# prediction = model.predict(new_data)
# print("Precio estimado por noche:", prediction[0])


## 7. Archivos generados

Al ejecutar el notebook se crearán:

- `models/airbnb_price_model.joblib`: modelo o pipeline entrenado.
- `models/airbnb_price_model_metadata.json`: parámetros, columnas, versiones y métricas.

No se guardan automáticamente los datos de entrenamiento. Para desplegar el modelo, mantené la estructura de columnas y las transformaciones consistentes con las usadas durante el entrenamiento.